In [42]:
import pandas as pd
import numpy as np

def read_pv_year_from_pvsam_csv(path, year=0, pv_col="ac_gross_kw"):
    """
    PVsAM output file contains 25 years of hourly data concatenated.
    year=0 -> first year (hours 0..8759), year=1 -> second year, etc.
    Returns: pv array length 8760 (kW), clipped to >=0.
    """
    df = pd.read_csv(path)

    if pv_col not in df.columns:
        raise ValueError(f"Column '{pv_col}' not found. Available: {list(df.columns)}")

    start = year * 8760
    end = start + 8760
    if end > len(df):
        raise ValueError(f"Requested year={year} exceeds file length ({len(df)} rows).")

    pv = df[pv_col].iloc[start:end].to_numpy(dtype=float)
    pv = np.clip(pv, 0.0, None)  # remove small negative night values
    return pv


In [ ]:
import pyomo.environ as pyo

def solve_pv_battery_milp(pv, price, demand,
                         dt=1.0,
                         Emax_kWh=50.0,
                         Pchg_max_kW=25.0,
                         Pdis_max_kW=25.0,
                         eta_chg=0.95,
                         eta_dis=0.95,
                         soc0_kWh=10.0,
                         soc_final_equal=True,
                         # paper-style constraints
                         soc_min_frac=0.20,         # EBESS,min = 0.2*EBESS,max  (DoD<=80%)
                         ncycles_max=1.0):          # Ncycles,max over horizon (e.g., 1 cycle/day)

    """
    Updated battery constraints following Dias de Lima et al. (2025) structure:
      - Single binary xBESS to prevent simultaneous charge/discharge (Eq.7-8)
      - SOC (EBESS_t) dynamics with efficiencies (Eq.9)
      - SOC bounds EBESS,min/EBESS,max (Eq.10-11)
      - Charge/discharge rates RBESS,ch and RBESS,disch (Eq.12-13)
      - Cycle count Ncycles = sum_t RBESS,disch (Eq.14)
      - Cycle limit Ncycles <= Ncycles,max (Eq.15)

    Note: We keep PV curtailment (pv_spill) to guarantee feasibility when PV exceeds demand+charge.
    """

    T = len(pv)
    assert len(price) == T and len(demand) == T, "pv, price, demand must have same length"

    m = pyo.ConcreteModel()
    m.T = pyo.RangeSet(0, T - 1)

    # Parameters
    m.pv     = pyo.Param(m.T, initialize={t: float(pv[t])     for t in range(T)})
    m.price  = pyo.Param(m.T, initialize={t: float(price[t])  for t in range(T)})
    m.demand = pyo.Param(m.T, initialize={t: float(demand[t]) for t in range(T)})

    # Scalars
    m.EBESS_max = pyo.Param(initialize=float(Emax_kWh))
    m.EBESS_min = pyo.Param(initialize=float(soc_min_frac) * float(Emax_kWh))
    m.PBESS_max = pyo.Param(initialize=float(max(Pchg_max_kW, Pdis_max_kW)))  # converter limit
    m.Ncycles_max = pyo.Param(initialize=float(ncycles_max))

    # Decision variables
    m.p_grid    = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.pv_spill  = pyo.Var(m.T, domain=pyo.NonNegativeReals)

    m.PBESS_ch    = pyo.Var(m.T, domain=pyo.NonNegativeReals)  # Eq.(7)
    m.PBESS_disch = pyo.Var(m.T, domain=pyo.NonNegativeReals)  # Eq.(8)
    m.EBESS       = pyo.Var(m.T, domain=pyo.NonNegativeReals)  # Eq.(9)-(11)

    # Single binary to avoid simultaneous charge/discharge (paper uses xBESS)
    m.xBESS = pyo.Var(m.T, domain=pyo.Binary)

    # Optional: keep separate hard limits for charge/discharge power (matches your original)
    # If you want exactly the paper (single PBESS,max), set both to PBESS_max.
    m.Pchg_max = pyo.Param(initialize=float(Pchg_max_kW))
    m.Pdis_max = pyo.Param(initialize=float(Pdis_max_kW))

    # Charge/discharge power limits with one binary (Eq.7-8 style)
    m.chg_limit = pyo.Constraint(m.T, rule=lambda m,t: m.PBESS_ch[t]    <= m.xBESS[t] * m.Pchg_max)
    m.dis_limit = pyo.Constraint(m.T, rule=lambda m,t: m.PBESS_disch[t] <= (1 - m.xBESS[t]) * m.Pdis_max)

    # Power balance (no export), with PV curtailment
    m.power_balance = pyo.Constraint(
        m.T,
        rule=lambda m,t: m.p_grid[t] + (m.pv[t] - m.pv_spill[t]) + m.PBESS_disch[t]
                         == m.demand[t] + m.PBESS_ch[t]
    )
    m.spill_cap = pyo.Constraint(m.T, rule=lambda m,t: m.pv_spill[t] <= m.pv[t])

    # Battery energy balance (Eq.9)
    def bess_energy_balance(m, t):
        inflow  = eta_chg * m.PBESS_ch[t] * dt
        outflow = (1/eta_dis) * m.PBESS_disch[t] * dt
        if t == 0:
            return m.EBESS[t] == soc0_kWh + inflow - outflow
        return m.EBESS[t] == m.EBESS[t-1] + inflow - outflow
    m.bess_energy_balance = pyo.Constraint(m.T, rule=bess_energy_balance)

    # SOC bounds (Eq.10-11)
    m.soc_min = pyo.Constraint(m.T, rule=lambda m,t: m.EBESS[t] >= m.EBESS_min)
    m.soc_max = pyo.Constraint(m.T, rule=lambda m,t: m.EBESS[t] <= m.EBESS_max)

    # Charge/discharge rate definitions (Eq.12-13)
    # RBESS,ch_t  = (eta * PBESS,ch_t * dt) / EBESS,max
    # RBESS,disch = ((1/eta) * PBESS,disch_t * dt) / EBESS,max
    m.RBESS_ch    = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.RBESS_disch = pyo.Var(m.T, domain=pyo.NonNegativeReals)

    m.rate_ch_def = pyo.Constraint(
        m.T, rule=lambda m,t: m.RBESS_ch[t] == (eta_chg * m.PBESS_ch[t] * dt) / m.EBESS_max
    )
    m.rate_dis_def = pyo.Constraint(
        m.T, rule=lambda m,t: m.RBESS_disch[t] == ((1/eta_dis) * m.PBESS_disch[t] * dt) / m.EBESS_max
    )

    # Cycle counting and limit (Eq.14-15)
    m.Ncycles = pyo.Var(domain=pyo.NonNegativeReals)
    m.cycles_def = pyo.Constraint(expr=m.Ncycles == sum(m.RBESS_disch[t] for t in m.T))
    m.cycles_limit = pyo.Constraint(expr=m.Ncycles <= m.Ncycles_max)

    # Optional: end SOC = initial SOC (daily cycle)
    if soc_final_equal:
        m.soc_end = pyo.Constraint(expr=m.EBESS[T-1] == soc0_kWh)

    # Objective: minimize grid energy cost
    m.obj = pyo.Objective(expr=sum(m.price[t] * m.p_grid[t] * dt for t in m.T), sense=pyo.minimize)

    solver = pyo.SolverFactory("gurobi")
    res = solver.solve(m, tee=False)

    return {
        "p_grid":   [pyo.value(m.p_grid[t]) for t in range(T)],
        "p_chg":    [pyo.value(m.PBESS_ch[t]) for t in range(T)],
        "p_dis":    [pyo.value(m.PBESS_disch[t]) for t in range(T)],
        "soc":      [pyo.value(m.EBESS[t]) for t in range(T)],
        "pv_spill": [pyo.value(m.pv_spill[t]) for t in range(T)],
        "Ncycles":  pyo.value(m.Ncycles),
        "cost":     pyo.value(m.obj),
        "solver_status": str(res.solver.status),
        "termination": str(res.solver.termination_condition),
    }


In [ ]:
PV_PATH = r"pvsam_ac_outputs.csv"  # your uploaded file path
pv = read_pv_year_from_pvsam_csv(PV_PATH, year=0, pv_col="ac_gross_kw")  # 8760 values

# TODO: replace these with your real hourly arrays (length 8760)
price  = [0.20] * 8760     # currency/kWh
demand = [5.0]  * 8760     # kW

result = solve_pv_battery_milp(
    pv=pv, price=price, demand=demand,
    Emax_kWh=50, Pchg_max_kW=25, Pdis_max_kW=25,
    soc0_kWh=10, soc_final_equal=True
)

df = pd.DataFrame({
    "p_grid":   result["p_grid"],
    "p_chg":    result["p_chg"],
    "p_dis":    result["p_dis"],
    "soc":      result["soc"],
    "pv_spill": result["pv_spill"],
})

print(df.head(120))
print("\nNcycles:", result["Ncycles"])
print("Total cost:", result["cost"])




    p_grid  p_chg  p_dis   soc  pv_spill
0      5.0    0.0    0.0  10.0       0.0
1      5.0    0.0    0.0  10.0       0.0
2      5.0    0.0    0.0  10.0       0.0
3      5.0    0.0    0.0  10.0       0.0
4      5.0    0.0    0.0  10.0       0.0
..     ...    ...    ...   ...       ...
95     5.0    0.0    0.0  10.0       0.0
96     5.0    0.0    0.0  10.0       0.0
97     5.0    0.0    0.0  10.0       0.0
98     5.0    0.0    0.0  10.0       0.0
99     5.0    0.0    0.0  10.0       0.0

[100 rows x 5 columns]

Ncycles: 1.0
Total cost: 5441.896453716858


In [47]:
print(df.head(130))
print("\nNcycles:", result["Ncycles"])
print("Total cost:", result["cost"])


       p_grid  p_chg  p_dis   soc  pv_spill
0    5.000000    0.0    0.0  10.0       0.0
1    5.000000    0.0    0.0  10.0       0.0
2    5.000000    0.0    0.0  10.0       0.0
3    5.000000    0.0    0.0  10.0       0.0
4    5.000000    0.0    0.0  10.0       0.0
..        ...    ...    ...   ...       ...
125  5.000000    0.0    0.0  10.0       0.0
126  5.000000    0.0    0.0  10.0       0.0
127  5.000000    0.0    0.0  10.0       0.0
128  4.868470    0.0    0.0  10.0       0.0
129  2.913514    0.0    0.0  10.0       0.0

[130 rows x 5 columns]

Ncycles: 1.0
Total cost: 5441.896453716858
